In [9]:
# !gcloud auth application-default login

In [10]:
import google.auth

_, project_id = google.auth.default()
print(f"Project ID: {project_id}")

Project ID: dev-mq-tech-transfer


In [11]:
import os as _os
import google.auth
from google.auth.transport.requests import Request
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace import TracerProvider, SpanLimits
from opentelemetry.sdk.trace.export import BatchSpanProcessor, SpanExporter, SpanExportResult
from opentelemetry.sdk.resources import Resource, SERVICE_NAME as _SVC_NAME, SERVICE_INSTANCE_ID as _SVC_INST
from openinference.instrumentation.google_adk import GoogleADKInstrumentor

_base_creds, _trace_project = google.auth.default()
_base_creds.refresh(Request())

# Retrieved context captured during each agent run (for evaluation).
_RETRIEVED_CONTEXT: list = []

_otel_resource = Resource.create({
    _SVC_NAME: "gap-assessment-agent",
    _SVC_INST: f"worker-{_os.getpid()}",
    "gcp.project_id": _trace_project,
})


class _OTLPRefreshingExporter(SpanExporter):
    """HTTP OTLP → telemetry.googleapis.com with auto-refreshing ADC token."""

    def __init__(self, creds, project_id):
        self._creds      = creds
        self._project_id = project_id
        self._req        = Request()

    def export(self, spans):
        from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter as _HTTPExp
        if not self._creds.valid:
            self._creds.refresh(self._req)
        return _HTTPExp(
            endpoint="https://telemetry.googleapis.com/v1/traces",
            headers={
                "Authorization":       f"Bearer {self._creds.token}",
                "x-goog-user-project": self._project_id,
            },
            timeout=10,
        ).export(spans)

    def shutdown(self):
        pass


tracer_provider = TracerProvider(
    resource=_otel_resource,
    span_limits=SpanLimits(
        max_span_attributes=64,
        max_span_attribute_length=4096,
    ),
)
tracer_provider.add_span_processor(BatchSpanProcessor(_OTLPRefreshingExporter(_base_creds, _trace_project)))
otel_trace.set_tracer_provider(tracer_provider)

_os.environ.setdefault("OTEL_SEMCONV_STABILITY_OPT_IN", "gen_ai_latest_experimental")
_os.environ.setdefault("GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY", "true")
_os.environ.setdefault("OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT", "EVENT_ONLY")

GoogleADKInstrumentor().instrument(tracer_provider=tracer_provider)
try:
    from opentelemetry.instrumentation.google_genai import GoogleGenAiSdkInstrumentor
    GoogleGenAiSdkInstrumentor().instrument(tracer_provider=tracer_provider)
    print("  + GoogleGenAiSdkInstrumentor active (gen_ai.* spans → Inputs/Outputs tab)")
except Exception as _e:
    print(f"  GoogleGenAiSdkInstrumentor skipped: {_e}")


def push_eval_metrics(scores: dict, attributes: dict):
    """Write eval scores to Cloud Monitoring as custom gauge time series."""
    import time as _t
    from google.cloud import monitoring_v3

    client = monitoring_v3.MetricServiceClient(credentials=_base_creds)
    project_name = f"projects/{_trace_project}"
    now = _t.time()
    seconds = int(now)
    nanos   = int((now - seconds) * 1e9)

    def _make_series(metric_name: str, value: float) -> monitoring_v3.TimeSeries:
        labels = {}
        for k, v in attributes.items():
            if isinstance(v, str):
                safe_key = k.replace(".", "_").replace("-", "_")
                labels[safe_key] = str(v)[:1024]
        return monitoring_v3.TimeSeries({
            "metric": {
                "type":   f"custom.googleapis.com/{metric_name}",
                "labels": labels,
            },
            "resource": {
                "type":   "global",
                "labels": {"project_id": _trace_project},
            },
            "points": [{
                "interval": {"end_time": {"seconds": seconds, "nanos": nanos}},
                "value":    {"double_value": float(value)},
            }],
        })

    series = [
        _make_series("rag/faithfulness",     scores.get("faithfulness", 0.0)),
        _make_series("rag/answer_relevancy",  scores.get("answer_relevancy", 0.0)),
    ]
    try:
        client.create_time_series(name=project_name, time_series=series)
        print(f"  Metrics → Cloud Monitoring (custom.googleapis.com/rag/*):")
        print(f"    rag/faithfulness     = {scores.get('faithfulness', 0):.2f}")
        print(f"    rag/answer_relevancy = {scores.get('answer_relevancy', 0):.2f}")
        print(f"  View: https://console.cloud.google.com/monitoring/metrics-explorer?project={_trace_project}")
    except Exception as _me:
        print(f"  [metrics] Cloud Monitoring write failed: {_me}")


print(f"\n✓ Tracing active — backend: Cloud Trace via OTLP (telemetry.googleapis.com)")
print(f"  View traces: https://console.cloud.google.com/traces/list?project={_trace_project}")
print(f"✓ Metrics active — custom.googleapis.com/rag/faithfulness & rag/answer_relevancy")


Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


  + GoogleGenAiSdkInstrumentor active (gen_ai.* spans → Inputs/Outputs tab)

✓ Tracing active — backend: Cloud Trace via OTLP (telemetry.googleapis.com)
  View traces: https://console.cloud.google.com/traces/list?project=dev-mq-tech-transfer
✓ Metrics active — custom.googleapis.com/rag/faithfulness & rag/answer_relevancy


In [12]:
import json
import os
from typing import Any, Dict, List

import vertexai
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.api_core.client_options import ClientOptions
from google.cloud import logging as gcp_logging
from google.cloud import discoveryengine_v1 as discoveryengine
from google.genai import types as genai_types


In [13]:
def load_prompt(prompt_id: str, project_id: str, location: str = "us-central1") -> str:
    """Load the latest version of the prompt from Vertex AI Prompt Management."""
    vai_client = vertexai.Client(project=project_id, location=location)
    loaded = vai_client.prompts.get(prompt_id=prompt_id)
    si = loaded.prompt_data.system_instruction
    if isinstance(si, str):
        text = si
    elif hasattr(si, "text"):
        text = si.text
    elif hasattr(si, "parts"):
        text = "".join(getattr(pt, "text", "") for pt in si.parts)
    else:
        text = str(si)
    print(f"Prompt loaded from Vertex AI (id: {prompt_id}, {len(text)} chars)")
    return text


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _log_step(
    project_id: str, step: str, user_id: str, query: str, response: str, citation: Any
) -> None:
    payload = {
        "Step": step,
        "User_id": user_id,
        "User_query": query,
        "Response": response,
        "Citation": citation,
    }
    gcp_logging.Client(project=project_id).logger("rag_workflow").log_struct(
        payload, severity="INFO"
    )


def _chunk_search_passages(
    query: str,
    project_id: str,
    location: str,
    datastore_id: str,
    page_size: int = 8,
) -> List[Dict[str, Any]]:
    """Chunk-mode search returning passage dicts."""
    endpoint = (
        "discoveryengine.googleapis.com"
        if location.lower() == "global"
        else f"{location}-discoveryengine.googleapis.com"
    )
    serving_config = (
        f"projects/{project_id}/locations/{location}/collections/default_collection/"
        f"dataStores/{datastore_id}/servingConfigs/default_config"
    )
    spec = discoveryengine.SearchRequest.ContentSearchSpec(
        search_result_mode=discoveryengine.SearchRequest.ContentSearchSpec.SearchResultMode.CHUNKS,
    )
    req = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=page_size,
        content_search_spec=spec,
    )

    passages: List[Dict[str, Any]] = []
    client = discoveryengine.SearchServiceClient(
        client_options=ClientOptions(api_endpoint=endpoint)
    )
    for r in client.search(req).results:
        chunk = getattr(r, "chunk", None)
        if chunk is None:
            continue
        content = str(getattr(chunk, "content", "") or "").strip()
        if not content:
            continue
        doc_meta = getattr(chunk, "document_metadata", None)
        title = str(getattr(doc_meta, "title", "") or "") if doc_meta else ""
        uri = str(getattr(doc_meta, "uri", "") or "") if doc_meta else ""
        chunk_name = str(getattr(chunk, "name", "") or "")
        parts = chunk_name.split("/")
        doc_id = parts[-3] if len(parts) >= 3 else ""
        passages.append(
            {"doc_id": doc_id, "title": title, "uri": uri, "content": content}
        )
    return passages


def make_search_tool(project_id: str, location: str, datastore_id: str):
    """Return an ADK-compatible search tool."""

    def search_datastore(query: str) -> str:
        """Search the document datastore for relevant chunks matching the query.

        Args:
            query: The search query to find relevant document chunks.

        Returns:
            Formatted document chunks with citation IDs, titles, URIs, and content.
        """
        passages = _chunk_search_passages(query, project_id, location, datastore_id)

        if not passages:
            return "No relevant documents found."

        lines = []
        for idx, p in enumerate(passages, 1):
            lines.append(
                f"[C{idx}]\n"
                f"title: {p['title']}\n"
                f"uri: {p['uri']}\n"
                f"content: {p['content']}"
            )
        result = "\n\n".join(lines)
        _RETRIEVED_CONTEXT.append(result)
        return result

    return search_datastore


# ---------------------------------------------------------------------------
# Agent runner
# ---------------------------------------------------------------------------

async def run_rag_agent(
    query: str,
    user_id: str,
    project_id: str,
    location: str,
    datastore_id: str,
    prompt_id: str,
    model_location: str = "us-central1",
) -> tuple:
    """Run the RAG agent and return (response_text, retrieved_context_text)."""
    global _RETRIEVED_CONTEXT
    _RETRIEVED_CONTEXT = []

    vertexai.init(project=project_id, location=model_location)
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
    os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
    os.environ["GOOGLE_CLOUD_LOCATION"] = model_location

    instruction = load_prompt(prompt_id, project_id, model_location)
    search_tool = make_search_tool(project_id, location, datastore_id)

    agent = Agent(
        model="gemini-2.5-flash",
        name="rag_agent",
        instruction=instruction,
        tools=[search_tool],
    )

    session_service = InMemorySessionService()
    session = await session_service.create_session(app_name="rag_app", user_id=user_id)
    runner = Runner(agent=agent, app_name="rag_app", session_service=session_service)

    _log_step(project_id, "agent_start", user_id, query, "", [])

    result_text = ""
    user_message = genai_types.Content(
        role="user",
        parts=[genai_types.Part.from_text(text=query)],
    )
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=user_message,
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        result_text = part.text
                        break

    _log_step(project_id, "agent_complete", user_id, query, result_text, [])
    context_text = "\n\n---\n\n".join(_RETRIEVED_CONTEXT)
    return result_text, context_text


In [ ]:
import asyncio
from functools import partial
from google import genai as google_genai
from ragas.llms import llm_factory
from ragas.llms.base import InstructorLLM
from ragas.embeddings import GoogleEmbeddings
from ragas.metrics.collections import Faithfulness, AnswerRelevancy

# RAGAS Collections metrics always call agenerate() internally, but
# instructor.from_genai() produces a sync client (is_async=False).
# Bridge: run generate() in a thread executor so ascore() can await it.
_orig_agenerate = InstructorLLM.agenerate

async def _agenerate_sync_safe(self, prompt, response_model):
    if not self.is_async:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, partial(self.generate, prompt, response_model))
    return await _orig_agenerate(self, prompt, response_model)

InstructorLLM.agenerate = _agenerate_sync_safe


def evaluate_rag_response(
    query: str,
    response: str,
    context: str,
    project_id: str,
    model_location: str = "us-central1",
) -> dict:
    genai_client = google_genai.Client(
        vertexai=True,
        project=project_id,
        location=model_location,
    )

    vertex_llm = llm_factory(
        "gemini-2.5-flash",
        provider="google",
        client=genai_client,
    )
    vertex_embeddings = GoogleEmbeddings(
        model="text-embedding-004",
        use_vertex=True,
        client=genai_client,
    )

    context_chunks = [c.strip() for c in context.split("\n\n---\n\n") if c.strip()]
    if not context_chunks:
        context_chunks = [context] if context.strip() else ["(no context retrieved)"]

    f_score = Faithfulness(llm=vertex_llm).score(
        user_input=query,
        response=response,
        retrieved_contexts=context_chunks,
    )
    ar_score = AnswerRelevancy(llm=vertex_llm, embeddings=vertex_embeddings).score(
        user_input=query,
        response=response,
    )

    return {
        "faithfulness": float(f_score),
        "answer_relevancy": float(ar_score),
        "reasoning": f"RAGAS | faithfulness={f_score:.3f}  answer_relevancy={ar_score:.3f}",
    }

In [15]:
# Set the prompt ID obtained from register_prompt.ipynb
REGISTERED_PROMPT_ID = "1651941728820658176"

SAMPLE_QUERY        = "tell me about the gaps in facilities"
TARGET_DATASTORE_ID = "masked-data_1781603945004"

_tracer = otel_trace.get_tracer("gap-assessment-agent")
with _tracer.start_as_current_span("rag_pipeline") as _pipeline_span:
    _pipeline_span.set_attribute("rag.query",        SAMPLE_QUERY[:500])
    _pipeline_span.set_attribute("rag.datastore_id", TARGET_DATASTORE_ID)
    _pipeline_span.set_attribute("vai.prompt_id",    REGISTERED_PROMPT_ID)

    result, retrieved_context = await run_rag_agent(
        query=SAMPLE_QUERY,
        user_id="sample-user-001",
        project_id=project_id,
        location="us",
        datastore_id=TARGET_DATASTORE_ID,
        prompt_id=REGISTERED_PROMPT_ID,
    )
    print("─" * 60)
    print(result[:500] + "..." if len(result) > 500 else result)
    print(f"\nContext chunks retrieved: {retrieved_context.count('[C')}")

    print("\nEvaluating response quality (Gemini as judge)...")
    with _tracer.start_as_current_span("rag.evaluation") as _eval_span:
        _eval_span.set_attribute("vai.prompt_id",    REGISTERED_PROMPT_ID)
        _eval_span.set_attribute("rag.query",        SAMPLE_QUERY[:500])
        _eval_span.set_attribute("rag.datastore_id", TARGET_DATASTORE_ID)
        scores = evaluate_rag_response(
            query=SAMPLE_QUERY,
            response=result,
            context=retrieved_context,
            project_id=project_id,
        )
        _eval_span.set_attribute("eval.faithfulness",     float(scores.get("faithfulness", 0)))
        _eval_span.set_attribute("eval.answer_relevancy", float(scores.get("answer_relevancy", 0)))
        _eval_span.set_attribute("eval.reasoning",        str(scores.get("reasoning", "")))

    _pipeline_span.set_attribute("eval.faithfulness",     float(scores.get("faithfulness", 0)))
    _pipeline_span.set_attribute("eval.answer_relevancy", float(scores.get("answer_relevancy", 0)))
    _pipeline_span.set_attribute("eval.reasoning",        str(scores.get("reasoning", "")))

    print(f"  faithfulness    : {scores.get('faithfulness', 0):.2f}")
    print(f"  answer_relevancy: {scores.get('answer_relevancy', 0):.2f}")
    print(f"  reasoning       : {scores.get('reasoning', '')}")

    print("\nPushing metrics to Cloud Monitoring...")
    push_eval_metrics(
        scores=scores,
        attributes={
            "rag_query":        SAMPLE_QUERY[:100],
            "rag_datastore_id": TARGET_DATASTORE_ID,
            "vai_prompt_id":    REGISTERED_PROMPT_ID,
        },
    )

tracer_provider.force_flush(timeout_millis=15_000)
print(f"\n✓ Traces  → https://console.cloud.google.com/traces/list?project={project_id}")
print(f"✓ Metrics → https://console.cloud.google.com/monitoring/metrics-explorer?project={project_id}")
print(f"            Metric prefix: custom.googleapis.com/rag/")


Prompt loaded from Vertex AI (id: 1651941728820658176, 3463 chars)
────────────────────────────────────────────────────────────
```json
[
  {
    "facility_area": "Facility design, capacity, and room classifications",
    "sending_site_requirements": [
      {
        "requirement": "Sterile filling isolator utilizing a rapid transfer port ([SENDING_SITE_5]) connection.",
        "source": "[C1]"
      },
      {
        "requirement": "RABS process with disc seals double bagged under unidirectional air flow (HEPA filtered air protection) and sterilized in an autoclave, then aseptically transferred onto the filling line....

Context chunks retrieved: 68

Evaluating response quality (Gemini as judge)...


c:\Users\L137844\Downloads\gap_assesment\tredence_gap_assessment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\L137844\AppData\Local\Temp\ipykernel_24828\4289184087.py:30: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  ChatVertexAI(
C:\Users\L137844\AppData\Local\Temp\ipykernel_24828\4289184087.py:30: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  ChatVertexAI(
C:\Users\L137844\AppData\Local\Temp\ipykernel_24828\4289184087.py:29: D

ValueError: Collections metrics only support modern InstructorLLM. Found: LangchainLLMWrapper. Use: llm_factory('gpt-4o-mini', client=openai_client)